# 05 — Apply Models

Applies trained models to a photometric catalog with Monte Carlo
error propagation to estimate stellar parameters and their uncertainties.

Large catalogs are processed in chunks to control memory usage.

**Workflow:**
1. Load catalog and rename columns to MINAS standard
2. Split into chunks and apply magnitude error filter
3. Load trained models
4. Run `Predictor` with Monte Carlo error propagation
5. Save results and show summary

**Output:** `results/<timestamp>_<SURVEY>_<MODEL_TYPE>.csv`

## Configuration

In [ ]:
import os
import glob
import shutil
import datetime
import numpy as np
import pandas as pd
import joblib as jb
import minas as mg
from minas.models import Predictor
from tqdm.contrib.concurrent import process_map

# Input catalog
INPUT_FILE = 'catalog.csv'
INDEX_COL  = None            # identifier column, or None

# Survey
SURVEY = 'SPLUS'

# Column mapping: raw catalog column names → renamed to MINAS standard
COORD_COLS  = ['RA', 'DEC']
FILTER_COLS = [
    'g_pstotal_0', 'i_pstotal_0', 'j0378_pstotal_0', 'j0395_pstotal_0',
    'j0410_pstotal_0', 'j0430_pstotal_0', 'j0515_pstotal_0',
    'j0660_pstotal_0', 'j0861_pstotal_0', 'r_pstotal_0', 'u_pstotal_0', 'z_pstotal_0',
]
ERROR_COLS = [
    'e_g_PStotal', 'e_i_PStotal', 'e_J0378_PStotal', 'e_J0395_PStotal',
    'e_J0410_PStotal', 'e_J0430_PStotal', 'e_J0515_PStotal',
    'e_J0660_PStotal', 'e_J0861_PStotal', 'e_r_PStotal', 'e_u_PStotal', 'e_z_PStotal',
]
OTHER_COLS = ['Dist']   # extra columns to load
KEEP_COLS  = []         # extra columns to carry over to the output CSV

# Model settings
MODEL_TYPE  = 'XGB'    # 'XGB' or 'RF'
MODELS_DIR  = 'models' # folder with trained models from 04_training.ipynb
TRAINING_ID = ''       # datetime string from 04_training.ipynb; empty = use most recent
PARAM_LIST  = ['teff', 'logg', 'feh']

# Distance column for absolute magnitude computation
DIST_COL = 'Dist'      # set to None if not available

# Magnitude error cut applied before prediction
MAG_ERR_CUT = 1.0

# Monte Carlo settings
MC_REPS          = 100  # total Monte Carlo realisations
BATCH_PARTITIONS = 10   # reps per batch (lower = less memory)
N_WORKERS        = 1    # parallel workers (>1 only useful for RF)

# Chunking for large catalogs
CHUNK_SIZE = 500_000

# Output
RESULTS_DIR  = 'results'

datetime_str = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Run ID     : {datetime_str}')
print(f'Model      : {MODEL_TYPE}  |  MC reps: {MC_REPS}  |  batch: {BATCH_PARTITIONS}')

## Step 1 — Load and rename catalog

In [ ]:
rename_dict  = dict(zip(COORD_COLS,  ['RA', 'DEC']))
rename_dict |= dict(zip(FILTER_COLS, mg.FILTERS[SURVEY]))
rename_dict |= dict(zip(ERROR_COLS,  mg.ERRORS[SURVEY]))

all_cols = COORD_COLS + FILTER_COLS + ERROR_COLS + OTHER_COLS
if INDEX_COL and INDEX_COL not in all_cols:
    all_cols = [INDEX_COL] + all_cols

catalog = mg.read_csv(INPUT_FILE, usecols=all_cols)
catalog = catalog.rename(columns=rename_dict)

print(f'Catalog loaded: {len(catalog):,} objects')

## Step 2 — Split into chunks and apply magnitude error filter

In [ ]:
TEMP_DIR = f'temp_{datetime_str}'
os.makedirs(TEMP_DIR, exist_ok=True)

mag_cols = mg.FILTERS[SURVEY]
err_cols = mg.ERRORS[SURVEY]
n_chunks = 0

for start in range(0, len(catalog), CHUNK_SIZE):
    chunk = catalog.iloc[start:start + CHUNK_SIZE].copy()

    for err_col in err_cols:
        if err_col in chunk.columns:
            chunk = chunk[chunk[err_col] <= MAG_ERR_CUT]

    if len(chunk) == 0:
        continue

    chunk.reset_index(drop=True).to_csv(f'{TEMP_DIR}/chunk_{n_chunks:03d}.csv', index=False)
    n_chunks += 1

print(f'Saved {n_chunks} chunks to {TEMP_DIR}/')

## Step 3 — Load trained models

In [ ]:
from minas.models import load_model_pipeline

ext       = 'json' if MODEL_TYPE == 'XGB' else 'sav'
model_dir = os.path.join(MODELS_DIR, MODEL_TYPE)
models    = {}

for param in PARAM_LIST:
    pattern = os.path.join(model_dir, f'*_{SURVEY}_{param}_{MODEL_TYPE}.{ext}')
    files   = sorted(glob.glob(pattern))

    if TRAINING_ID:
        files = [f for f in files if TRAINING_ID in f]

    if not files:
        raise FileNotFoundError(
            f'No model found for [{param}] in {model_dir}.\n'
            f'Run 04_training.ipynb first.'
        )

    models[param] = load_model_pipeline(MODEL_TYPE, files[-1])
    print(f'  [{param:4s}] {os.path.basename(files[-1])}')

print('\nAll models loaded.')

## Step 4 — Run Predictor with Monte Carlo error propagation

In [ ]:
predictor = Predictor(
    id_col          = INDEX_COL,
    mag_cols        = mag_cols,
    err_cols        = err_cols,
    dist_col        = DIST_COL,
    correction_pairs= None,
    models          = models,
    mc_reps         = MC_REPS,
    batch_partitions= BATCH_PARTITIONS,
)

output_path = os.path.join(RESULTS_DIR, f'{datetime_str}_{SURVEY}_{MODEL_TYPE}.csv')
chunk_paths = sorted(glob.glob(f'{TEMP_DIR}/chunk_*.csv'))

# Write header
pd.DataFrame(columns=
    ['RA', 'DEC']
    + [f'{p}{s}' for p in models for s in ['', '-ERR']]
    + KEEP_COLS
).to_csv(output_path, index=False)

for i, chunk_path in enumerate(chunk_paths):
    print(f'Chunk {i+1}/{len(chunk_paths)}...')
    chunk = pd.read_csv(chunk_path)
    args  = (chunk, output_path, KEEP_COLS, 'a', False)

    if MODEL_TYPE == 'RF' and N_WORKERS > 1:
        process_map(predictor.predict_parameters, [args], max_workers=N_WORKERS, total=1)
    else:
        predictor.predict_parameters(args)

shutil.rmtree(TEMP_DIR)
print(f'\nDone. Results saved to: {output_path}')